In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2
import zlib
import base64

In [56]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = "USR_GSIT_GCHIARELLA"
pw = "F#AnhHvHZ0N_15"
hostname = "172.20.0.18"
service_name="tramite"
port = 1521

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

In [57]:
query1 = """SELECT TRUNC(tp.fecha_creacion, 'MONTH') AS mes, COUNT(*) AS cantidad_documentos
FROM TRAMITE_NUEVO.TATD_TRAMITE_PRINCIPAL tp
WHERE tp.fecha_creacion >= TO_DATE('2023-01-01', 'YYYY-MM-DD')
GROUP BY TRUNC(tp.fecha_creacion, 'MONTH')
ORDER BY mes ASC"""
query2 = """SELECT TRUNC(tp.fecha_registro, 'MONTH') AS mes, COUNT(*) AS cantidad_documentos
FROM TRAMITE_NUEVO.TATD_TRAMITE_SEGUIMIENTO tp
WHERE tp.fecha_registro >= TO_DATE('2023-01-01', 'YYYY-MM-DD')
GROUP BY TRUNC(tp.fecha_registro, 'MONTH')
ORDER BY mes ASC"""

In [58]:
documentos1 = pd.read_sql_query(query1, con=connection0)
documentos2 = pd.read_sql_query(query2, con=connection0)

In [59]:
documentos1

,mes,cantidad_documentos
0,2023-01-01,179377
1,2023-02-01,157498
2,2023-03-01,197064
3,2023-04-01,165063
4,2023-05-01,173580
5,2023-06-01,166073
6,2023-07-01,160954
7,2023-08-01,168737
8,2023-09-01,179534
9,2023-10-01,170765


In [60]:
documentos2

,mes,cantidad_documentos
0,2023-01-01,956051
1,2023-02-01,810322
2,2023-03-01,989165
3,2023-04-01,837100
4,2023-05-01,945218
5,2023-06-01,878517
6,2023-07-01,833303
7,2023-08-01,907905
8,2023-09-01,917054
9,2023-10-01,907906


In [65]:
resultado = pd.merge(documentos1, documentos2, on='mes', suffixes=('1', '2'))
resultado['cantidad_total'] = resultado['cantidad_documentos1'] + resultado['cantidad_documentos2']

In [66]:
resultado

,mes,cantidad_documentos1,cantidad_documentos2,cantidad_total
0,2023-01-01,179377,956051,1135428
1,2023-02-01,157498,810322,967820
2,2023-03-01,197064,989165,1186229
3,2023-04-01,165063,837100,1002163
4,2023-05-01,173580,945218,1118798
5,2023-06-01,166073,878517,1044590
6,2023-07-01,160954,833303,994257
7,2023-08-01,168737,907905,1076642
8,2023-09-01,179534,917054,1096588
9,2023-10-01,170765,907906,1078671


In [67]:
resultado['cantidad_hojas'] = resultado['cantidad_total'] * 14.72

In [68]:
resultado

,mes,cantidad_documentos1,cantidad_documentos2,cantidad_total,cantidad_hojas
0,2023-01-01,179377,956051,1135428,16713500.16
1,2023-02-01,157498,810322,967820,14246310.40
2,2023-03-01,197064,989165,1186229,17461290.88
3,2023-04-01,165063,837100,1002163,14751839.36
4,2023-05-01,173580,945218,1118798,16468706.56
5,2023-06-01,166073,878517,1044590,15376364.80
6,2023-07-01,160954,833303,994257,14635463.04
7,2023-08-01,168737,907905,1076642,15848170.24
8,2023-09-01,179534,917054,1096588,16141775.36
9,2023-10-01,170765,907906,1078671,15878037.12


In [76]:
resultado.to_csv(f'cantidad.csv', index=False)

In [69]:
resultado_nuevo = resultado[resultado['mes']<'2024-01-01']

In [70]:
resultado_nuevo

,mes,cantidad_documentos1,cantidad_documentos2,cantidad_total,cantidad_hojas
0,2023-01-01,179377,956051,1135428,16713500.16
1,2023-02-01,157498,810322,967820,14246310.40
2,2023-03-01,197064,989165,1186229,17461290.88
3,2023-04-01,165063,837100,1002163,14751839.36
4,2023-05-01,173580,945218,1118798,16468706.56
5,2023-06-01,166073,878517,1044590,15376364.80
6,2023-07-01,160954,833303,994257,14635463.04
7,2023-08-01,168737,907905,1076642,15848170.24
8,2023-09-01,179534,917054,1096588,16141775.36
9,2023-10-01,170765,907906,1078671,15878037.12


In [73]:
suma = resultado_nuevo['cantidad_hojas'].sum()

In [74]:
suma

189274661.76000002

In [6]:
base.tail()

,id_archivo,content_type,fecha_documento,contenido_archivo
4712,818694,application/pdf,2021-01-29,b'%PDF-1.3\n%\xe2\xe3\xcf\xd3\n29 0 obj\n<<\n/...
4713,818697,application/pdf,2021-01-29,b'%PDF-1.3\n%\xe2\xe3\xcf\xd3\n29 0 obj\n<<\n/...
4714,818685,application/pdf,2021-01-29,b'%PDF-1.3\n%\xe2\xe3\xcf\xd3\n10 0 obj\n<<\n/...
4715,822651,application/pdf,2021-01-29,b'%PDF-1.4\r%\xe2\xe3\xcf\xd3\r\n113 0 obj\r<<...
4716,391255,application/pdf,2021-01-30,b'%PDF-1.5\r%\xe2\xe3\xcf\xd3\r\n10 0 obj\r<</...


In [9]:
import io
import zipfile
from PIL import Image
from docx import Document
from pdfminer.high_level import extract_text

In [14]:
from io import BytesIO
import zipfile
from PIL import Image
import docx
from pdfminer.high_level import extract_text

def obtener_numero_hojas(contenido_archivo, content_type):
  """
  Obtains the number of sheets/pages from various file types.

  Args:
      contenido_archivo (bytes): The file content in binary format.
      content_type (str): The file's content type (e.g., application/pdf).

  Returns:
      int: The number of sheets/pages in the file.
  """

  numero_hojas = 0

  if content_type == "application/pdf":
    # Extract text using pdfminer for more reliable page counting
    try:
      with BytesIO(contenido_archivo) as buffer:
        texto_pdf = extract_text(buffer)
      numero_hojas = texto_pdf.count("\f")  # Count form feed characters
    except Exception as e:
      print(f"Error extracting text from PDF: {e}")  # Handle potential errors gracefully

  elif content_type == "application/msword":
    # Open the DOCX file from content
    try:
      with BytesIO(contenido_archivo) as buffer:
        documento_docx = docx.Document(buffer)
      numero_hojas = len(documento_docx.sections)  # Count sections (equivalent to sheets)
    except Exception as e:
      print(f"Error processing DOCX file: {e}")

  elif content_type.startswith("image/"):
    # Open the image
    try:
      with BytesIO(contenido_archivo) as buffer:
        imagen = Image.open(buffer)
      numero_hojas = imagen.n_frames if hasattr(imagen, 'n_frames') else 1
    except Exception as e:
      print(f"Error processing image file: {e}")

  return numero_hojas


In [15]:
base["numero_hojas"] = base.apply(lambda row: obtener_numero_hojas(row["contenido_archivo"], row["content_type"]), axis=1)


Error processing DOCX file: file '<_io.BytesIO object at 0x7537dde105e0>' is not a Word file, content type is 'application/vnd.openxmlformats-officedocument.themeManager+xml'
Error processing DOCX file: file '<_io.BytesIO object at 0x7537dc77d1c0>' is not a Word file, content type is 'application/vnd.openxmlformats-officedocument.themeManager+xml'
Error processing DOCX file: file '<_io.BytesIO object at 0x7537dd281440>' is not a Word file, content type is 'application/vnd.openxmlformats-officedocument.themeManager+xml'
Error processing DOCX file: file '<_io.BytesIO object at 0x7537de4ae5c0>' is not a Word file, content type is 'application/vnd.openxmlformats-officedocument.themeManager+xml'
Error processing DOCX file: File is not a zip file
Error processing DOCX file: file '<_io.BytesIO object at 0x7537dd2309a0>' is not a Word file, content type is 'application/vnd.openxmlformats-officedocument.themeManager+xml'
Error processing DOCX file: File is not a zip file
Error processing DOCX f

In [17]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4717 entries, 0 to 4716
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_archivo         4717 non-null   int64         
 1   content_type       4717 non-null   object        
 2   fecha_documento    4717 non-null   datetime64[ns]
 3   contenido_archivo  4717 non-null   object        
 4   numero_hojas       4717 non-null   int64         
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 184.4+ KB


In [19]:
base.head()

,id_archivo,content_type,fecha_documento,contenido_archivo,numero_hojas
0,379622,application/pdf,2021-01-01,b'%PDF-1.4\n%\xb0\xb8\xba\x95\n1 0 obj<</Type/...,1
1,376349,application/vnd.openxmlformats-officedocument....,2021-01-02,"b'PK\x03\x04\x14\x00\x08\x08\x08\x00\x92\x95""R...",0
2,379621,application/pdf,2021-01-02,b'%PDF-1.5\r%\xe2\xe3\xcf\xd3\r\n21 0 obj\r<</...,5
3,577838,application/vnd.openxmlformats-officedocument....,2021-01-03,b'PK\x03\x04\x14\x00\x06\x00\x08\x00\x00\x00!\...,0
4,386449,application/pdf,2021-01-04,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n1 0 obj\n<< \n/...,25


In [26]:
base

,id_archivo,content_type,fecha_documento,contenido_archivo,numero_hojas,mes,primer_dia_mes
0,379622,application/pdf,2021-01-01,b'%PDF-1.4\n%\xb0\xb8\xba\x95\n1 0 obj<</Type/...,3.655925,1,2021-01-01
1,376349,application/vnd.openxmlformats-officedocument....,2021-01-02,"b'PK\x03\x04\x14\x00\x08\x08\x08\x00\x92\x95""R...",3.655925,1,2021-01-01
2,379621,application/pdf,2021-01-02,b'%PDF-1.5\r%\xe2\xe3\xcf\xd3\r\n21 0 obj\r<</...,5.000000,1,2021-01-01
3,577838,application/vnd.openxmlformats-officedocument....,2021-01-03,b'PK\x03\x04\x14\x00\x06\x00\x08\x00\x00\x00!\...,3.655925,1,2021-01-01
4,386449,application/pdf,2021-01-04,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n1 0 obj\n<< \n/...,25.000000,1,2021-01-01
...,...,...,...,...,...,...,...
4712,818694,application/pdf,2021-01-29,b'%PDF-1.3\n%\xe2\xe3\xcf\xd3\n29 0 obj\n<<\n/...,23.000000,1,2021-01-01
4713,818697,application/pdf,2021-01-29,b'%PDF-1.3\n%\xe2\xe3\xcf\xd3\n29 0 obj\n<<\n/...,23.000000,1,2021-01-01
4714,818685,application/pdf,2021-01-29,b'%PDF-1.3\n%\xe2\xe3\xcf\xd3\n10 0 obj\n<<\n/...,4.000000,1,2021-01-01
4715,822651,application/pdf,2021-01-29,b'%PDF-1.4\r%\xe2\xe3\xcf\xd3\r\n113 0 obj\r<<...,5.000000,1,2021-01-01


In [24]:
# Calcular el promedio de hojas totales por documento
promedio_hojas_por_documento = base['numero_hojas'].sum() / base['id_archivo'].nunique()

# Remplazar valores de numero_hojas que sean 0 o 1 por el promedio calculado
base['numero_hojas'] = base['numero_hojas'].apply(lambda x: promedio_hojas_por_documento if x in [0, 1] else x)

# Convertir la columna 'fecha_documento' a tipo datetime
base['fecha_documento'] = pd.to_datetime(base['fecha_documento'])

# Agregar una columna 'primer_dia_mes' que contenga el primer día de cada mes
base['primer_dia_mes'] = base['fecha_documento'].dt.to_period('M').dt.to_timestamp()

# Agrupar por 'primer_dia_mes' y sumar la cantidad de hojas y contar el número de documentos
nuevo_df = base.groupby('primer_dia_mes').agg({'numero_hojas': 'sum', 'id_archivo': 'count'}).reset_index()

# Renombrar las columnas
nuevo_df.columns = ['mes', 'cantidad_hojas', 'cantidad_documentos']

# Mostrar el nuevo DataFrame
print(nuevo_df)

         mes  cantidad_hojas  cantidad_documentos
0 2021-01-01    21896.535722                 4717


In [25]:
nuevo_df

,mes,cantidad_hojas,cantidad_documentos
0,2021-01-01,21896.535722,4717
